# SAR in Colab | Understand feature interactions, then run GPT-2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSAE/SAE-Behavioral-Compositionality/blob/main/notebooks/SAR_Quickstart.ipynb)

**No local setup, no API key.** In Colab choose **Runtime → Run all**. A GPU is helpful but not required.

**Question:** if two SAE features have known individual effects, do those effects predict what happens when both are kept? SAR measures the variation that a model using only individual effects leaves unexplained.

This notebook first uses a tiny fully transparent example, then performs **real SAE feature interventions** on GPT-2 Small. It uses the exact core estimator in the paper, not a reimplementation.

The large-model results in the paper are **not** reproduced by this small demonstration.

## 1 · Load the core code (and set up Colab)

The `sar.py` implementation remains untouched. A small `sar_runner.py` handles model/SAE adapters and preserves the original reconstruction residual.

In [ ]:
# Environment and project code. On Colab, install known compatible versions.
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
if IS_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformer-lens==3.5.0", "sae-lens==6.44.4"])

# A Colab notebook opened from GitHub does not automatically contain repo files.
REPO_URL = "https://github.com/OpenSAE/SAE-Behavioral-Compositionality.git"
project = next((p for p in [Path.cwd(), Path.cwd().parent]
                if (p / "sar_runner.py").exists() and (p / "sar.py").exists()), None)
if project is None:
    project = Path("/content/SAE-Behavioral-Compositionality" if IS_COLAB else "./SAE-Behavioral-Compositionality")
    if not (project / "sar_runner.py").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(project)])
    if not (project / "sar_runner.py").exists():
        raise FileNotFoundError("sar_runner.py missing. Push the new files to the public repository first.")
sys.path.insert(0, str(project.resolve()))

import numpy as np
from sar import coalitions, design, sar
from sar_runner import (ActivationSite, FunctionalModelAdapter, FunctionalSAE,
                        HookedTransformerAdapter, LogitDifference, TorchSAEFeatures,
                        evaluate_sae)
print("SAR source:", project)
print("NumPy:", np.__version__)


## 2 · Why two features can have a joint effect

The next example has six features and exactly one interaction. The downstream behavior depends on a product of two feature values. An additive first-order model cannot express that product, while a pairwise model can.

In [ ]:
# A transparent, model-free intervention game: one real interaction.
masks, fit, held = coalitions(K=6, M=384, n_fit=256, seed=0)
s = masks.astype(float)
g = s @ np.array([1.0, 0.5, 1.0, -0.3, 0.2, 0.1]) - 1.0 * s[:, 0] * s[:, 2]
r = sar(g, masks, fit, held)
print(f"First-order SAR: {r['sar1']:.3f}")
print(f"Pairwise SAR:    {r['sar2']:.6f}")
print(f"Pairwise share:  {r['rho2']:.3f}")
print(f"Held-out rank fidelity: {r['rank1']:.3f} -> {r['rank2']:.3f}")
assert r['sar1'] > 0.01 and r['sar2'] < 1e-16

## 3 · Use a publicly released SAE on a real language model

We use a pretrained GPT-2 SAE from SAE Lens. The chosen layer and hook **must match** the SAE checkpoint. The forward pass resumes from `hook_resid_pre` after replacing the activation; unselected features and the reconstruction residual stay untouched.

In [ ]:
# Public GPT-2 Small + the matching published SAE checkpoint.
# Download requires internet access but no Hugging Face authentication.
import torch
from sae_lens import SAE
from transformer_lens import HookedTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "gpt2"
SAE_RELEASE = "gpt2-small-res-jb"
HOOK_NAME = "blocks.8.hook_resid_pre"

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()
pretrained = SAE.from_pretrained(release=SAE_RELEASE, sae_id=HOOK_NAME, device=DEVICE)
pretrained = pretrained[0] if isinstance(pretrained, tuple) else pretrained
pretrained.eval()
sae_features = TorchSAEFeatures(pretrained)
assert pretrained.W_dec.shape[1] == model.cfg.d_model
print("Model:", MODEL_NAME, "  SAE site:", HOOK_NAME,
      "  Dictionary:", pretrained.W_dec.shape, "  Device:", DEVICE)


## 4 · Compute SAR from actual downstream model behavior

For each prompt, we take the top 8 active SAE features, sample 144 coalitions, fit first-order and pairwise models on 104 and test them on 40 *held-out samples*. The scalar behavior is the logit difference between the expected next name and an alternative. These small prompt examples are for exploration, **not** paper-level conclusions.

In [ ]:
# Feel free to change these input prompts or K/M after the first run.
CASES = [
    ("Alice Emma Alice", " Emma", " Rachel"),
    ("Peter Steven Peter", " Steven", " Nina"),
]
K, M, N_FIT, BATCH = 8, 144, 104, 16
experiments = []
for seed, (prompt, target, foil) in enumerate(CASES):
    runner = HookedTransformerAdapter(
        model, HOOK_NAME, LogitDifference(target, foil), run_mode="resume")
    result = evaluate_sae(runner, sae_features, prompt,
                          K=K, M=M, n_fit=N_FIT, seed=seed, batch_size=BATCH)
    experiments.append((prompt, result))
    m = result.metrics
    print(f"{prompt!r}: SAR1={m['sar1']:.4f}  SAR2={m['sar2']:.4f}  "
          f"rank={m['rank1']:.3f} -> {m['rank2']:.3f}  "
          f"top feature IDs={result.feature_ids.tolist()}")
assert len(experiments) == len(CASES)


## 5 · Plot the held-out predictions

The points below are coalitions **not used to fit** either model. Better predictions lie closer to the diagonal. Small demonstrations may vary by prompt and sampling seed.

In [ ]:
import matplotlib.pyplot as plt

prompt, result = experiments[0]
hold = result.held_indices
y_true = result.responses[hold]
preds = {}
for order in [1, 2]:
    X = design(result.masks, order)
    coefficients = np.linalg.lstsq(X[result.fit_indices],
                                  result.responses[result.fit_indices], rcond=None)[0]
    preds[order] = X[hold] @ coefficients
for order, label in [(1, "Individual feature effects"), (2, "Individual + pairwise effects")]:
    plt.figure(figsize=(5.5, 4.5))
    plt.scatter(y_true, preds[order], alpha=0.65, s=26)
    lo = min(y_true.min(), preds[order].min())
    hi = max(y_true.max(), preds[order].max())
    plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
    plt.title(f"{label} | rank {result.metrics['rank'+str(order)]:.3f}")
    plt.xlabel("Actual downstream logit difference")
    plt.ylabel("Predicted logit difference")
    plt.tight_layout()
    plt.show()


## 6 · Use your own model or SAE

The demonstration does not require editing the original `demo/demo.py`. To evaluate your **own SAE**, use the companion notebook:

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSAE/SAE-Behavioral-Compositionality/blob/main/notebooks/Use_Your_Own_SAE.ipynb)

For the reference implementation see [`sar_runner.py`](../sar_runner.py). **Important:** SAR is relative to a chosen feature subset, token, intervention protocol, behavior and coalition distribution; it is not a single global score for a complete SAE dictionary.